# Phase-field fracture of a notched square plate

## A PhAST laboratory for Google Colab

Build a dynamic fracture calculation with the public PhAST solver, from specimen
geometry and mesh generation to field plots and animations. The exercises reuse
the same geometry, configuration and post-processing functions.

The specimen is a single-edge-notched tension (SENT) plate of soda-lime glass,
following the configuration used by Borden et al. (2012). The plate is pulled
apart on its top and bottom edges. A diffuse crack grows from the notch tip and
crosses the remaining ligament.

## Learning objectives

On completing this laboratory you should be able to:

1. Describe the AT2 phase-field model and the role of the length scale $\ell_0$.
2. Construct a parametric specimen geometry and generate a graded mesh.
3. Connect named mesh regions to displacement boundary conditions.
4. Explain why an explicit dynamic route imposes a CFL restriction on the time step.
5. Distinguish the mechanics subproblem from the damage subproblem in a staggered scheme.
6. Reload stored fields through the public result interface and animate them.
7. Design and run your own parameter study, starting from an inclined notch.

## Scope of the example

This classroom example uses a coarse mesh and one run per configuration to
illustrate the model and software workflow. Mesh and time-step convergence
studies are needed before using it for quantitative benchmark comparisons.

## Software

All mechanics and all damage physics are executed inside PhAST
([CEMS-Lab/PhAST](https://github.com/CEMS-Lab/PhAST)). Gmsh generates the mesh.
NumPy and Matplotlib derive scalar views of stored fields and render them.
The notebook configures the public solver and post-processes its stored fields.

<div class="badge-row">
<a class="badge-colab" href="https://colab.research.google.com/github/CEMS-Lab/autumn-school/blob/main/notebooks/study/classroom/01_simulate_fracture.ipynb">Open in Colab (published edition)</a> · <a class="badge-link" href="https://cems-lab.github.io/autumn-school/notebooks/study/classroom/01_simulate_fracture.ipynb">Download notebook</a> · <a class="badge-link" href="https://cems-lab.github.io/autumn-school/notebooks/solutions/classroom/01_simulate_fracture.ipynb">Download with recap answer</a> · <a class="badge-link" href="https://cems-lab.github.io/autumn-school/SETUP.md">Environment setup</a>
</div>


**Day 2 edition · 15 September 2026.** Run the notebook to generate its plots and animations; this edition includes the code without retained run outputs. Complete the reference before trying the additional parameter studies.

## 1. Prepare the Colab environment

PhAST requires Python 3.10 or newer, together with compatible dependencies. The
setup cell installs the public package when PhAST is unavailable. Use a fresh
runtime and record the installed version and resolved commit with your results.

`PHAST_REF` selects a branch or tag; an empty value selects the default branch.
Branch names identify a moving source, so record the resolved commit for
reproducibility.

Colab also needs `libglu1-mesa` for Gmsh. Measure installation separately from
the complete notebook, including mesh generation, simulation, plotting and
animation export. Fresh whole-notebook Colab timing remains to be recorded for
this edition.


In [ ]:
import importlib.util
import subprocess
import sys

try:
    IN_COLAB = importlib.util.find_spec("google.colab") is not None
except ModuleNotFoundError:
    IN_COLAB = False

print("Python:", sys.version.split()[0])
print("Running on Colab:", IN_COLAB)

if IN_COLAB:
    # Gmsh's Python wheel needs libGLU at import time.
    subprocess.run(["apt-get", "-qq", "install", "-y", "libglu1-mesa"], check=True)

PHAST_REPOSITORY = "https://github.com/CEMS-Lab/PhAST.git"
PHAST_REF = ""   # "" for the default branch, or e.g. "v0.16.2" or a branch name

requirement = f"phast @ git+{PHAST_REPOSITORY}" + (f"@{PHAST_REF}" if PHAST_REF else "")
if importlib.util.find_spec("phast") is None:
    print("Installing:", requirement)
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", requirement], check=True)

import phast

# Importing PhAST selects the non-interactive Agg backend, which several of its
# batch runners require. Restore the inline backend afterwards so that figures
# and animation players appear in the notebook.
%matplotlib inline

import importlib.metadata
import matplotlib

print("PhAST is importable. Matplotlib backend:", matplotlib.get_backend())
print("PhAST version:", importlib.metadata.version("phast"))

The `doctor` subcommand reports the resolved environment, including the
interpreter, the PyTorch build, the accelerator status and the optional sparse
backends. Read it before interpreting any timing.

In [ ]:
!{sys.executable} -m phast doctor

## 2. The physical model in one page

The plate occupies a domain $\Omega$. Two fields are solved for: the
displacement $\mathbf{u}(\mathbf{x},t)$ and the scalar phase field
$d(\mathbf{x},t) \in [0,1]$, where $d = 0$ denotes intact material and $d = 1$
denotes a fully broken state.

**Regularised energy.** The AT2 model represents a sharp crack by a diffuse band
of width controlled by the length scale $\ell_0$:

$$
\Psi = \int_\Omega \Big[ g(d)\,\psi^{+}(\boldsymbol{\varepsilon}) + \psi^{-}(\boldsymbol{\varepsilon}) \Big]\,\mathrm{d}\Omega
\;+\; \int_\Omega \frac{G_c}{2\ell_0}\Big[ d^{2} + \ell_0^{2}\,|\nabla d|^{2} \Big]\,\mathrm{d}\Omega ,
$$

with degradation function $g(d) = (1-d)^2 + \eta$ and a small residual stiffness
$\eta$ that keeps the stiffness matrix non-singular.

**Energy split.** The elastic energy density is separated by a spectral
decomposition of the strain into a tensile part $\psi^{+}$, which drives damage,
and a compressive part $\psi^{-}$, which does not. This prevents cracks from
forming under pure compression.

**Irreversibility.** A history variable enforces monotone damage growth:

$$
\mathcal{H}(\mathbf{x},t) = \max_{s \le t} \psi^{+}\big(\boldsymbol{\varepsilon}(\mathbf{x},s)\big).
$$

**Damage equation.** Stationarity of $\Psi$ with respect to $d$ gives a linear
elliptic problem for the AT2 model:

$$
\frac{G_c}{\ell_0}\,d - G_c \ell_0 \nabla^2 d + 2\mathcal{H}\,d = 2\mathcal{H}.
$$

**Mechanics.** Linear momentum balance is integrated in time with the lumped
mass matrix:

$$
\rho\,\ddot{\mathbf{u}} = \nabla\!\cdot\!\boldsymbol{\sigma}\big(\boldsymbol{\varepsilon},d\big).
$$

**Staggered solution.** Within each time step PhAST advances mechanics, updates
the history field, and then solves the damage equation. Mechanics is explicit;
the damage subproblem is solved implicitly by a matrix-free conjugate gradient
method. Each subproblem therefore uses a different numerical update.

**Mesh requirement.** The diffuse band must be resolved by the mesh. A working
rule is $h \le \ell_0 / 2$ inside the expected crack path, where $h$ is the
element size. The configuration below uses $\ell_0 = 0.5$ mm and a refined
element size of $0.25$ mm.

## 3. A parametric specimen geometry

The geometry is written as a Gmsh `.geo` script by the function below. The notch
inclination is a parameter, which allows the same function to serve both the
reference horizontal notch and the inclined-notch exercise in Section 12.

| Symbol | Meaning | Reference value |
|---|---|---|
| `L` | plate edge length | 40 mm |
| `a` | notch length measured from the left edge | 20 mm |
| `angle_deg` | notch inclination above the horizontal | 0 deg |
| `h_crack` | element size in the refined band | 0.25 mm |
| `h_coarse` | element size in the far field | 2.0 mm |
| `band` | half-width of the refined band | 1.5 mm |
| `eps` | half-opening of the notch mouth | 0.001 mm |

Six named physical curves are created: `top`, `bottom`, `left`, `right`,
`notch_upper` and `notch_lower`. Boundary conditions later refer to these names
rather than to coordinate tolerances.

Line 9 runs from the notch tip to the right edge. It carries mesh refinement
along the anticipated crack path and is treated as an internal interface between
the two plate surfaces, so it imposes no mechanical condition.

In [ ]:
import math
from pathlib import Path


def write_sent_geo(path, *, L=40.0, a=20.0, angle_deg=0.0,
                   h_coarse=2.0, h_crack=0.25, band=1.5, eps=1.0e-3):
    """Write a Gmsh .geo file for a square plate with a straight inclined notch.

    Parameters
    ----------
    path : str or Path
        Destination of the .geo file.
    L : float
        Plate edge length in mm.
    a : float
        Notch length measured from the left edge in mm.
    angle_deg : float
        Notch inclination above the horizontal, in degrees.
    h_coarse, h_crack : float
        Far-field and refined element sizes in mm.
    band : float
        Half-width of the refined band around the notch and expected path.
    eps : float
        Half-opening of the notch mouth on the left edge in mm.

    Returns
    -------
    Path
        The path that was written.
    """
    theta = math.radians(float(angle_deg))
    cy = L / 2.0
    tip_x = a * math.cos(theta)
    tip_y = cy + a * math.sin(theta)

    geo = f"""// PhAST teaching geometry: square plate with a notch at {angle_deg} deg
L = {L}; a = {a}; h_crack = {h_crack}; h_coarse = {h_coarse}; band = {band};

Point(1) = {{0, 0, 0, h_coarse}};              // bottom-left
Point(2) = {{L, 0, 0, h_coarse}};              // bottom-right
Point(3) = {{L, L, 0, h_coarse}};              // top-right
Point(4) = {{0, L, 0, h_coarse}};              // top-left
Point(5) = {{0, {cy + eps:.9g}, 0, h_crack}};  // notch mouth, upper lip
Point(6) = {{0, {cy - eps:.9g}, 0, h_crack}};  // notch mouth, lower lip
Point(7) = {{{tip_x:.9g}, {tip_y:.9g}, 0, h_crack}};   // notch tip
Point(8) = {{{L:.9g}, {tip_y:.9g}, 0, h_crack}};       // path exit on right edge

Line(1) = {{1, 2}};   // bottom
Line(2) = {{2, 8}};   // right, below the path exit
Line(3) = {{8, 3}};   // right, above the path exit
Line(4) = {{3, 4}};   // top
Line(5) = {{4, 5}};   // left, above the notch mouth
Line(6) = {{5, 7}};   // upper notch face
Line(7) = {{7, 6}};   // lower notch face
Line(8) = {{6, 1}};   // left, below the notch mouth
Line(9) = {{7, 8}};   // anticipated crack path, used for refinement

Curve Loop(1) = {{1, 2, -9, 7, 8}};
Plane Surface(1) = {{1}};
Curve Loop(2) = {{9, 3, 4, 5, 6}};
Plane Surface(2) = {{2}};

Physical Curve("bottom") = {{1}};
Physical Curve("right") = {{2, 3}};
Physical Curve("top") = {{4}};
Physical Curve("left") = {{5, 8}};
Physical Curve("notch_upper") = {{6}};
Physical Curve("notch_lower") = {{7}};
Physical Surface("plate") = {{1, 2}};

// Graded size field: h_crack near the notch faces and the anticipated path.
Field[1] = Distance;
Field[1].CurvesList = {{6, 7, 9}};
Field[1].Sampling = 100;
Field[2] = Threshold;
Field[2].InField = 1;
Field[2].SizeMin = h_crack;
Field[2].SizeMax = h_coarse;
Field[2].DistMin = band;
Field[2].DistMax = 3*band;
Background Field = 2;
Mesh.CharacteristicLengthExtendFromBoundary = 0;
Mesh.CharacteristicLengthFromPoints = 0;
Mesh.CharacteristicLengthFromCurvature = 0;
Mesh.Algorithm = 6;   // Frontal-Delaunay
Mesh.ElementOrder = 1;
"""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(geo, encoding="utf-8")
    return path


WORK = Path("phast_lab")
WORK.mkdir(exist_ok=True)

geo_file = write_sent_geo(WORK / "reference.geo", angle_deg=0.0)
print(geo_file.read_text()[:600])

## 4. Generate the mesh and recover the named regions

Gmsh is driven through its Python API, which avoids any dependency on a command
line binary being present in the runtime. The mesh is written in the legacy
`msh2` format, which carries the physical group names that PhAST reads.

`phast.FEMMesh` takes ownership of the mesh from this point: it stores the
connectivity, precomputes the element geometry and the lumped mass, and exposes
the named node sets.

In [ ]:
import gmsh


def build_mesh(geo_path, msh_path, *, verbose=False):
    """Mesh a .geo file with the Gmsh Python API and write msh2 output."""
    gmsh.initialize()
    try:
        gmsh.option.setNumber("General.Terminal", 1 if verbose else 0)
        gmsh.open(str(geo_path))
        gmsh.model.mesh.generate(2)
        gmsh.option.setNumber("Mesh.MshFileVersion", 2.2)
        gmsh.write(str(msh_path))
    finally:
        gmsh.finalize()
    return Path(msh_path)


mesh_file = build_mesh(geo_file, WORK / "reference.msh")

import phast

mesh = phast.FEMMesh(str(mesh_file), device="cpu")
print(mesh.summary())
print("Named node sets:", sorted(mesh.node_sets.keys()))

The plot below shows the mesh and the four loaded or constrained boundaries. The
grading is visible as a refined band along the notch and along the anticipated
crack path.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.tri import Triangulation

nodes = np.asarray(mesh.nodes.cpu(), dtype=float)
elements = np.asarray(mesh.elements.cpu(), dtype=int)
triangulation = Triangulation(nodes[:, 0], nodes[:, 1], elements)

fig, ax = plt.subplots(figsize=(6.4, 6.0))
ax.triplot(triangulation, lw=0.25, color="0.75")
styles = {
    "top": ("#336b87", "prescribed u_y = +0.002 mm"),
    "bottom": ("#336b87", "prescribed u_y = -0.002 mm"),
    "left": ("#7a8288", "u_x fixed"),
    "right": ("#7a8288", "u_x fixed"),
    "notch_upper": ("#a94f42", "traction-free notch face"),
    "notch_lower": ("#a94f42", "traction-free notch face"),
}
seen = set()
for name, (colour, label) in styles.items():
    idx = np.asarray(mesh.node_sets[name])
    shown = label if label not in seen else None
    seen.add(label)
    ax.scatter(nodes[idx, 0], nodes[idx, 1], s=8, color=colour, label=shown, zorder=3)
ax.set_aspect("equal")
ax.set_xlabel("x [mm]")
ax.set_ylabel("y [mm]")
ax.set_title(f"Reference mesh: {mesh.n_nodes} nodes, {mesh.n_elems} T3 elements")
ax.legend(loc="lower left", fontsize=8, framealpha=0.9)
plt.show()

## 5. Material and the stable time step

The material is the soda-lime glass of the reference benchmark. Units are
millimetres, newtons, megapascals and tonnes per cubic millimetre, which form a
consistent set.

| Property | Symbol | Value |
|---|---|---|
| Young's modulus | $E$ | 32 000 MPa |
| Poisson's ratio | $\nu$ | 0.20 |
| Density | $\rho$ | $2.45 \times 10^{-9}$ t/mm³ |
| Critical energy release rate | $G_c$ | $3.0 \times 10^{-3}$ N/mm |
| Phase-field length scale | $\ell_0$ | 0.5 mm |
| Residual stiffness | $\eta$ | $10^{-7}$ |

`phast.Material` resolves the constitutive data and reports the dilatational
wave speed $c_p$. The explicit central-difference integrator is stable only when

$$\Delta t \le \alpha\,\frac{h_\min}{c_p},$$

with a safety factor $\alpha$ of 0.8. PhAST applies this rule internally from the
mesh that is actually supplied; the cell below reproduces the arithmetic so that
the cost of refining the mesh is visible.

In [ ]:
material = phast.Material(
    E=32000.0,            # Young's modulus [MPa]
    nu=0.20,              # Poisson's ratio
    rho=2.45e-9,          # density [tonne/mm^3]
    Gc=3.0e-3,            # critical energy release rate [N/mm]
    l0=0.5,               # phase-field length scale [mm]
    eta_residual=1.0e-7,  # residual stiffness
    pf_model="AT2",
    energy_split="spectral",
    plane_stress=False,   # plane strain
)

h_min = float(mesh.h_min)
c_p = float(material.c_p)
dt_stable = 0.8 * h_min / c_p

print(f"h_min          = {h_min:.6f} mm")
print(f"c_p            = {c_p:.4e} mm/s")
print(f"stable dt      = {dt_stable:.4e} s")
print(f"l0 / h_min     = {material.l0 / h_min:.2f}   (values above 2 resolve the band)")

## 6. Loading, boundary conditions and the solver route

PhAST reads a YAML configuration. The builder below returns that configuration
as a Python dictionary so that every exercise can vary one entry and rerun.

**Boundary conditions.** The top and bottom edges receive equal and opposite
prescribed vertical displacements, which produces a symmetric mode-I opening
without translating the specimen. The left and right edges are restrained
horizontally. The notch faces carry no condition and remain traction-free.

**Loading.** A smooth-step ramp raises the prescribed displacement to its full
amplitude over 20 microseconds and holds it afterwards. A smooth ramp limits the
spurious stress waves that a sudden application would introduce.

**Route.** Two questions are decided separately. Whether inertia is retained is a
physical question, answered here by choosing a dynamic analysis. Whether a given
subproblem is advanced explicitly or implicitly is a numerical question, answered
here by explicit central-difference mechanics combined with an implicit linear
solve for the damage field.

| Mechanics | Damage | Status in this notebook |
|---|---|---|
| Dynamic, explicit central difference | Classical implicit AT2 solve | Used here |
| Quasi-static equilibrium | Classical implicit damage solve | Supported, separate examples |
| Dynamic, explicit | Learned proposal or audited replacement | Experimental, see Section 13 |

**Simulated window.** The reference run covers 50 microseconds, which is
sufficient for the crack to cross the ligament. Set `T_TOTAL = 1.0e-4` for the
longer window used in the published example.

**Snapshot cadence.** `h5_every` controls how often a full field snapshot is
written to the Zarr trajectory store. Every 20th step gives a smooth animation
at a modest storage cost.

In [ ]:
T_TOTAL = 5.0e-5       # simulated window [s]
SNAPSHOT_EVERY = 20    # store a full field snapshot every N steps


def make_config(mesh_path, *, name="SENT reference", E=32000.0, nu=0.20,
                Gc=3.0e-3, l0=0.5, rho=2.45e-9, eta_residual=1.0e-7,
                u_amp=0.002, t_total=T_TOTAL, t_ramp=2.0e-5,
                dt_safety=0.8, snapshot_every=SNAPSHOT_EVERY):
    """Return a PhAST run configuration for the notched square plate."""
    return {
        "schema_version": 1,
        "problem": {"name": name},
        "geometry": {"units": "mm", "mesh_path": str(mesh_path)},
        "material": {
            "E": E, "nu": nu, "Gc": Gc, "l0": l0, "rho": rho,
            "eta_residual": eta_residual,
            "energy_split": "spectral", "pf_model": "AT2",
        },
        "boundary_conditions": [
            {"nodes": "left",   "type": "fix",       "component": 0},
            {"nodes": "right",  "type": "fix",       "component": 0},
            {"nodes": "top",    "type": "prescribe", "component": 1, "value":  u_amp},
            {"nodes": "bottom", "type": "prescribe", "component": 1, "value": -u_amp},
        ],
        "loading": {
            "protocol": "simple", "ramp_type": "smooth_step",
            "t_ramp": t_ramp, "t_total": t_total,
        },
        "solver": {"solver_type": "explicit", "dt_safety": dt_safety},
        "output": {
            "trajectory": True, "trajectory_format": "zarr",
            "h5_every": snapshot_every, "fast": True, "print_every": 500,
        },
    }


import yaml

config = make_config(mesh_file)
config_file = WORK / "reference.yaml"
config_file.write_text(yaml.safe_dump(config, sort_keys=False), encoding="utf-8")
print(config_file.read_text())

The `precheck` subcommand reports wave speeds, the stable increment and mesh
quality without advancing the solution. Run it before committing to a long
calculation.

In [ ]:
!{sys.executable} -m phast precheck --config {config_file}

## 7. Run the simulation

The solver runs through PhAST's command-line interface. Progress messages report
the step, physical time and maximum damage. Record the elapsed time on your runtime.


In [ ]:
import subprocess
import sys
import time

run_dir = WORK / "runs" / "reference"
command = [
    sys.executable, "-m", "phast", "run", str(config_file),
    "--device", "cpu", "--output_dir", str(run_dir),
]
print(" ".join(command))

start = time.perf_counter()
process = subprocess.run(command, capture_output=True, text=True)
elapsed = time.perf_counter() - start
print(process.stdout[-2500:])
if process.returncode != 0:
    print(process.stderr[-2500:])
    raise RuntimeError("PhAST run failed")
print(f"Wall-clock time on this runtime: {elapsed:.1f} s")

## 8. A reusable post-processing toolkit

Everything below the solver is post-processing. The `RunViewer` class collects
the operations that the exercises need: listing the stored snapshots, projecting
element fields onto nodes, plotting a single frame, and animating a sequence.

Two storage conventions matter. Displacement, velocity and damage are nodal
fields with one value per node. Strain and stress are element fields with one
constant value per triangle. `phast.compute_field` performs the projection from
elements to nodes and returns a display label and a colour map alongside the
values, so derived quantities stay consistent between static plots and
animations.

Available field names are `damage`, `displacement_mag`, `H`, and any key of
`phast.FIELD_REGISTRY`, which includes `stress_xx`, `stress_yy`, `stress_xy`,
`von_mises_stress`, `max_principal_stress`, `min_principal_stress`,
`hydrostatic_stress`, `stress_triaxiality`, `strain_xx`, `strain_yy`,
`strain_xy`, `von_mises_strain`, `max_principal_strain` and
`min_principal_strain`.

In [ ]:
from matplotlib.animation import FuncAnimation, PillowWriter
import torch
import zarr


class RunViewer:
    """Read-only view of a finished PhAST run, prepared for plotting.

    The whole trajectory is read from the Zarr store once, in bulk, and derived
    fields are cached as arrays of shape (n_snapshots, n_nodes). Animation then
    updates the colour array of a single drawing rather than rebuilding the plot
    on every frame, which keeps a live classroom session responsive.

    Examples
    --------
    >>> view = RunViewer(run_dir)
    >>> view.plot_field("damage")
    >>> view.animate_field("von_mises_stress", stride=4)
    """

    #: fields read in bulk from the trajectory store
    _RAW_KEYS = ("step", "time_s", "displacement", "stress", "strain",
                 "damage_nodal", "H_nodal", "velocity")

    def __init__(self, run_dir, *, nu=0.20):
        self.run_dir = Path(run_dir)
        self.result = phast.load_result(self.run_dir)
        self.nu = float(nu)
        self.mesh = phast.FEMMesh(str(self.run_dir / "mesh.msh"), device="cpu")
        self.nodes = np.asarray(self.mesh.nodes.cpu(), dtype=float)
        self.elements = np.asarray(self.mesh.elements.cpu(), dtype=int)
        self.triangulation = Triangulation(
            self.nodes[:, 0], self.nodes[:, 1], self.elements)
        self._raw = self._load_trajectory()
        self.steps = self._raw["step"]
        self.times = self._raw["time_s"]
        self._series_cache = {}

    # -- bulk read ---------------------------------------------------------
    def _load_trajectory(self):
        """Read every stored snapshot in one pass over the Zarr store."""
        root = zarr.open(str(self.run_dir / "training_data.zarr"), mode="r")
        trajectory = root["simulation_data"]["trajectory"]
        count = int(trajectory.attrs.get("count", len(trajectory["step"])))
        raw = {key: np.asarray(trajectory[key][:count])
               for key in self._RAW_KEYS if key in trajectory}
        raw["step"] = raw["step"].astype(int)
        raw["time_s"] = raw["time_s"].astype(float)
        return raw

    def index_of(self, step):
        """Position of a stored step number inside the snapshot arrays."""
        return int(np.clip(np.searchsorted(self.steps, int(step)),
                           0, len(self.steps) - 1))

    def time_of(self, step):
        """Physical time in seconds for a stored step number."""
        return float(self.times[self.index_of(step)])

    # -- derived nodal fields ----------------------------------------------
    def series(self, field_name):
        """Return (values, label, colour map) with values of shape (frames, nodes).

        Accepts ``damage``, ``displacement_mag``, ``H`` and any key of
        ``phast.FIELD_REGISTRY``. Element fields are projected onto the nodes by
        ``phast.compute_field``. Results are cached, so a second call is free.
        """
        if field_name in self._series_cache:
            return self._series_cache[field_name]

        if field_name == "displacement_mag":
            values = np.linalg.norm(self._raw["displacement"], axis=2)
            label, cmap = "Displacement magnitude [mm]", "viridis"
        else:
            frames = len(self.steps)
            values = np.empty((frames, self.nodes.shape[0]), dtype=float)
            label = cmap = None
            for i in range(frames):
                stress = self._raw["stress"][i].astype(np.float64)
                strain = self._raw["strain"][i].astype(np.float64)
                frame, label, cmap = phast.compute_field(
                    field_name,
                    tuple(torch.as_tensor(stress[:, k]) for k in range(3)),
                    tuple(torch.as_tensor(strain[:, k]) for k in range(3)),
                    mesh=self.mesh,
                    d=torch.as_tensor(self._raw["damage_nodal"][i].astype(np.float64)),
                    H=torch.as_tensor(self._raw["H_nodal"][i].astype(np.float64)),
                    nu=self.nu)
                values[i] = np.asarray(torch.as_tensor(frame).cpu(), dtype=float)

        self._series_cache[field_name] = (values, label, cmap)
        return self._series_cache[field_name]

    def limits(self, field_name, values=None):
        """Colour limits held fixed across every frame."""
        if field_name == "damage":
            return 0.0, 1.0
        if values is None:
            values, _, _ = self.series(field_name)
        low, high = float(np.nanmin(values)), float(np.nanmax(values))
        return low, max(high, low + 1.0e-12)

    def frame(self, field_name, step=-1):
        """Nodal values of one field at one stored step."""
        values, _, _ = self.series(field_name)
        index = len(self.steps) - 1 if step == -1 else self.index_of(step)
        return values[index]

    def deformed(self, step, scale):
        """Triangulation on the displaced configuration, magnified by ``scale``."""
        if not scale:
            return self.triangulation
        index = len(self.steps) - 1 if step == -1 else self.index_of(step)
        moved = self.nodes + scale * self._raw["displacement"][index]
        return Triangulation(moved[:, 0], moved[:, 1], self.elements)

    # -- single frame -------------------------------------------------------
    def plot_field(self, field_name, step=-1, *, ax=None, vmin=None, vmax=None,
                   deform_scale=0.0, colorbar=True, title=None, shading="gouraud"):
        """Draw one field at one stored step."""
        values, label, cmap = self.series(field_name)
        index = len(self.steps) - 1 if step == -1 else self.index_of(step)
        low, high = self.limits(field_name, values[index][None, :])
        vmin = low if vmin is None else vmin
        vmax = high if vmax is None else vmax
        if ax is None:
            _, ax = plt.subplots(figsize=(5.4, 4.8))
        art = ax.tripcolor(self.deformed(self.steps[index], deform_scale),
                           values[index], shading=shading, cmap=cmap,
                           vmin=vmin, vmax=vmax)
        ax.set_aspect("equal")
        ax.set_xlabel("x [mm]")
        ax.set_ylabel("y [mm]")
        ax.set_title(title or f"{label}\nstep {self.steps[index]}, "
                              f"t = {self.times[index] * 1e6:.2f} us")
        if colorbar:
            ax.figure.colorbar(art, ax=ax, shrink=0.85)
        return ax

    # -- animations ----------------------------------------------------------
    def animate_field(self, field_name, *, stride=4, fps=12, gif_path=None,
                      figsize=(5.2, 4.7)):
        """Animate one field. Only the colour array changes between frames."""
        values, label, cmap = self.series(field_name)
        vmin, vmax = self.limits(field_name, values)
        values, times = values[::stride], self.times[::stride]

        fig, ax = plt.subplots(figsize=figsize)
        art = ax.tripcolor(self.triangulation, values[0], shading="gouraud",
                           cmap=cmap, vmin=vmin, vmax=vmax)
        fig.colorbar(art, ax=ax, shrink=0.85)
        ax.set_aspect("equal")
        ax.set_xlabel("x [mm]")
        ax.set_ylabel("y [mm]")
        title = ax.set_title("")

        def draw(i):
            art.set_array(values[i])
            title.set_text(f"{label}\nt = {times[i] * 1e6:.2f} us")
            return art, title

        animation = FuncAnimation(fig, draw, frames=len(times), interval=1000 / fps)
        if gif_path is not None:
            animation.save(str(gif_path), writer=PillowWriter(fps=fps))
        plt.close(fig)
        return animation

    def animate_panel(self, field_names, *, stride=4, fps=12, gif_path=None,
                      ncols=2, figsize=(9.0, 7.6)):
        """Animate several fields side by side on a shared time axis."""
        panels = []
        for name in field_names:
            values, label, cmap = self.series(name)
            vmin, vmax = self.limits(name, values)
            panels.append((values[::stride], label, cmap, vmin, vmax))
        times = self.times[::stride]

        nrows = int(np.ceil(len(field_names) / ncols))
        fig, axes = plt.subplots(nrows, ncols, figsize=figsize)
        axes = np.atleast_1d(axes).ravel()
        artists = []
        for (values, label, cmap, vmin, vmax), ax in zip(panels, axes):
            art = ax.tripcolor(self.triangulation, values[0], shading="gouraud",
                               cmap=cmap, vmin=vmin, vmax=vmax)
            fig.colorbar(art, ax=ax, shrink=0.82)
            ax.set_aspect("equal")
            ax.set_xticks([])
            ax.set_yticks([])
            ax.set_title(label, fontsize=10)
            artists.append(art)
        for ax in axes[len(field_names):]:
            ax.axis("off")
        heading = fig.suptitle("")

        def draw(i):
            for art, panel in zip(artists, panels):
                art.set_array(panel[0][i])
            heading.set_text(f"t = {times[i] * 1e6:.2f} microseconds")
            return artists

        animation = FuncAnimation(fig, draw, frames=len(times), interval=1000 / fps)
        if gif_path is not None:
            animation.save(str(gif_path), writer=PillowWriter(fps=fps))
        plt.close(fig)
        return animation


def crack_tip_history(view, *, threshold=0.9):
    """Return (time [us], furthest broken x [mm]) over the stored snapshots.

    A node counts as broken when its damage exceeds ``threshold``. The furthest
    such node in x is reported as the crack tip position.
    """
    damage, _, _ = view.series("damage")
    tips = np.full(len(view.steps), np.nan)
    for i in range(len(view.steps)):
        broken = damage[i] > threshold
        if broken.any():
            tips[i] = view.nodes[broken, 0].max()
    return view.times * 1e6, tips


view = RunViewer(run_dir)
print(f"{len(view.steps)} stored snapshots, "
      f"steps {view.steps[0]} to {view.steps[-1]}, "
      f"t up to {view.times[-1] * 1e6:.1f} us")
print("Stored fields:", view.result.field_names())

## 9. Final state

The four panels below show the state at the last stored snapshot. Read them
together. The displacement field shows the imposed opening and the discontinuity
across the crack. The vertical strain concentrates in the process zone. The von
Mises stress falls close to zero inside the broken band, which is the mechanical
signature of the degradation function. The phase field shows the diffuse crack
itself, with a band width set by $\ell_0$.

In [ ]:
fields = ("displacement_mag", "strain_yy", "von_mises_stress", "damage")

fig, axes = plt.subplots(2, 2, figsize=(11.0, 9.2))
for name, ax in zip(fields, axes.ravel()):
    view.plot_field(name, step=-1, ax=ax)
fig.suptitle(f"Final state, t = {view.times[-1] * 1e6:.1f} microseconds", fontsize=13)
fig.tight_layout()
plt.show()

The same fields may be drawn on the deformed configuration. The displacements
are of the order of a few micrometres, so a large magnification factor is needed
for the opening to be visible.

In [ ]:
fig, ax = plt.subplots(figsize=(6.0, 5.4))
view.plot_field("damage", step=-1, ax=ax, deform_scale=500.0,
                title="Damage on the deformed mesh, magnification 500")
plt.show()

## 10. Animate the four fields

The derived series were built once in Section 8 and cached, so the only work
left is rendering. Each frame updates the colour array of an existing drawing,
which is considerably cheaper than redrawing the contours.

`stride` selects every nth stored snapshot. Increasing it reduces the number
of frames to render. Set `SAVE_GIFS = True` to save GIF files for reports or
offline viewing, and include export time in the complete activity timing.

The animations are displayed inline with `to_jshtml`, which embeds a player with
frame controls.

In [ ]:
from IPython.display import HTML, display

STRIDE = 4        # render every 4th stored snapshot
SAVE_GIFS = False  # also write .gif files into the working directory

plt.rcParams["animation.embed_limit"] = 100  # MB of embedded animation per notebook

animations = {}
for name in fields:
    gif = WORK / f"reference_{name}.gif" if SAVE_GIFS else None
    animations[name] = view.animate_field(name, stride=STRIDE, fps=12, gif_path=gif)
print("Rendered:", list(animations))

In [ ]:
display(HTML(animations["displacement_mag"].to_jshtml()))

In [ ]:
display(HTML(animations["strain_yy"].to_jshtml()))

In [ ]:
display(HTML(animations["von_mises_stress"].to_jshtml()))

In [ ]:
display(HTML(animations["damage"].to_jshtml()))

A combined panel places the four fields on one time axis, which makes the
sequence of events easier to follow: the stress wave arrives first, the process
zone forms at the notch tip, and the damage band then advances while the stress
behind it collapses.

In [ ]:
panel = view.animate_panel(
    fields, stride=STRIDE, fps=12,
    gif_path=WORK / "reference_panel.gif" if SAVE_GIFS else None)
display(HTML(panel.to_jshtml()))

## 11. Energy and crack-tip histories

A dynamic result should be judged from histories as well as from final images.
PhAST writes `energy.csv` and related tables into the run directory, and they are
reachable through the public result interface.

The energy plot separates the stored elastic energy, the kinetic energy and the
dissipated fracture energy. The fracture energy rises as the crack advances. The
crack-tip trace is obtained from the toolkit function `crack_tip_history` and
gives an estimate of the propagation speed, which should be compared with the
Rayleigh wave speed of the material.

In [ ]:
print("Available histories:", view.result.history_names())

energy = view.result.history("energy")
columns = {key: np.array([row[key] for row in energy], dtype=float)
           for key in energy[0]}
print("Energy columns:", list(columns))

time_key = next(key for key in ("time", "t_s", "t") if key in columns)
time_us = columns[time_key] * 1e6

crack_time, crack_x = crack_tip_history(view)

fig, axes = plt.subplots(1, 2, figsize=(12.0, 4.4))
for name in ("elastic", "kinetic", "fracture", "external", "total"):
    if name in columns:
        axes[0].plot(time_us, columns[name], label=name)
axes[0].set_xlabel("Time [microseconds]")
axes[0].set_ylabel("Energy [N mm]")
axes[0].set_title("Energy history")
axes[0].grid(alpha=0.25)
axes[0].legend()

axes[1].plot(crack_time, crack_x, color="#a94f42")
axes[1].set_xlabel("Time [microseconds]")
axes[1].set_ylabel("Furthest broken node, x [mm]")
axes[1].set_title("Crack-tip position, damage threshold 0.9")
axes[1].grid(alpha=0.25)

fig.tight_layout()
plt.show()

valid = np.isfinite(crack_x)
if valid.sum() > 2:
    speed = np.gradient(crack_x[valid], crack_time[valid] * 1e-6)  # mm/s
    print(f"Mean propagation speed over the traced window: {np.nanmean(speed):.3e} mm/s")
    print(f"Dilatational wave speed c_p:                   {c_p:.3e} mm/s")

## Key takeaways

- Named mesh regions connect geometry to boundary conditions.
- Dynamic mechanics retains inertia and requires a stable time step.
- Fixed colour scales make evolving fields comparable.

### Consolidation

Why does refining the mesh increase dynamic computation time?

<details class="course-hint"><summary>Hint</summary><p>Consider the smallest element and the wave speed.</p></details>

<details class="course-solution"><summary>Conceptual answer</summary><p>The explicit stability limit scales with the smallest element dimension divided by wave speed. A smaller element reduces the stable time step and increases the number of updates over the same physical duration.</p></details>

## 12. Exercises

The exercises reuse `write_sent_geo`, `build_mesh`, `make_config` and
`RunViewer`. Each one asks for a short written answer as well as a calculation.
Record the wall-clock time of every run you report.

### Exercise 1 — Read the reference result

No new calculation is required.

1. State the element count, `h_min` and the stable increment of the reference mesh.
2. Measure the width of the damage band across the crack at $x = 30$ mm, taking
   the band edges where $d = 0.5$, and compare it with $\ell_0$.
3. Explain why the von Mises stress falls to near zero behind the crack tip while
   the displacement field remains finite there.

**Your answer:**

_Write here._

In [ ]:
# Exercise 1, part 2: a starting point.
damage = view.frame("damage", step=-1)
column = np.abs(view.nodes[:, 0] - 30.0) < 0.3
order = np.argsort(view.nodes[column, 1])
y = view.nodes[column, 1][order]
d = damage[column][order]

fig, ax = plt.subplots(figsize=(5.6, 4.0))
ax.plot(y, d, marker="o", ms=3)
ax.axhline(0.5, color="0.6", ls="--")
ax.set_xlabel("y [mm]")
ax.set_ylabel("Damage")
ax.set_title("Damage profile across the crack at x = 30 mm")
ax.grid(alpha=0.25)
plt.show()

### Exercise 2 — An inclined notch

Repeat the calculation with the notch inclined at 20 degrees above the
horizontal. The specimen is still pulled vertically, so the notch is no longer
aligned with the direction that a mode-I crack prefers.

1. Run the case with the cell below.
2. Compare the crack path with the reference result. Measure the angle at which
   the crack leaves the notch tip, and the angle it settles into further on.
3. Explain the kink using the tensile part of the strain energy: the crack turns
   towards the plane on which the maximum principal stress acts normally.
4. Repeat for 30 and 45 degrees. Report how the initiation time and the crack
   path change, and state at which inclination the refinement band supplied by
   `write_sent_geo` stops covering the actual path.

**Your answer:**

_Write here._

In [ ]:
def run_case(tag, *, angle_deg=0.0, geo_kwargs=None, config_kwargs=None):
    """Mesh, configure and run one variant. Returns a RunViewer for the result."""
    geo_kwargs = dict(geo_kwargs or {})
    config_kwargs = dict(config_kwargs or {})
    case_geo = write_sent_geo(WORK / f"{tag}.geo", angle_deg=angle_deg, **geo_kwargs)
    case_mesh = build_mesh(case_geo, WORK / f"{tag}.msh")
    case_config = make_config(case_mesh, name=tag, **config_kwargs)
    case_config_file = WORK / f"{tag}.yaml"
    case_config_file.write_text(yaml.safe_dump(case_config, sort_keys=False),
                                encoding="utf-8")
    case_dir = WORK / "runs" / tag

    start = time.perf_counter()
    outcome = subprocess.run(
        [sys.executable, "-m", "phast", "run", str(case_config_file),
         "--device", "cpu", "--output_dir", str(case_dir)],
        capture_output=True, text=True)
    elapsed = time.perf_counter() - start
    if outcome.returncode != 0:
        print(outcome.stdout[-2000:])
        print(outcome.stderr[-2000:])
        raise RuntimeError(f"Run {tag} failed")
    print(f"{tag}: finished in {elapsed:.1f} s")
    return RunViewer(case_dir)


inclined = run_case("notch_20deg", angle_deg=20.0)

fig, axes = plt.subplots(1, 2, figsize=(11.0, 4.6))
view.plot_field("damage", step=-1, ax=axes[0], title="Horizontal notch, final damage")
inclined.plot_field("damage", step=-1, ax=axes[1], title="20 degree notch, final damage")
fig.tight_layout()
plt.show()

In [ ]:
# Animate the inclined case. All four fields are available here as well.
inclined_panel = inclined.animate_panel(fields, stride=STRIDE, fps=12)
display(HTML(inclined_panel.to_jshtml()))

### Exercise 3 — The length scale and the mesh

The length scale $\ell_0$ sets the width of the diffuse band and, through the
regularised model, the effective strength of the material. The mesh must resolve
that width.

1. Rerun the reference case with $\ell_0 = 0.25$ mm and with $\ell_0 = 1.0$ mm,
   keeping the mesh fixed. Report the band width and the initiation time in each case.
2. State which of the three runs violates $h \le \ell_0 / 2$ inside the crack path,
   and describe what the damage field looks like when the requirement fails.
3. Rerun the $\ell_0 = 0.25$ mm case with `h_crack=0.1`. Report the change in
   element count, in the stable increment and in the wall-clock time, and explain
   why refining the mesh raises the cost more than proportionally.

**Your answer:**

_Write here._

In [ ]:
# Exercise 3: one variant is set up for you. Set RUN_THIS to True to execute it,
# then copy the cell and change the parameters for the remaining cases.
RUN_THIS = False

if RUN_THIS:
    coarse_l0 = run_case("l0_1p0", angle_deg=0.0, config_kwargs={"l0": 1.0})

    fig, axes = plt.subplots(1, 2, figsize=(11.0, 4.6))
    view.plot_field("damage", step=-1, ax=axes[0], title="l0 = 0.5 mm")
    coarse_l0.plot_field("damage", step=-1, ax=axes[1], title="l0 = 1.0 mm")
    fig.tight_layout()
    plt.show()
else:
    print("Set RUN_THIS = True to run the l0 = 1.0 mm variant "
          "(about one solve, comparable to Section 7).")

### Exercise 4 — Loading amplitude and rate

1. Rerun the reference case with `u_amp` reduced to 0.0015 mm and raised to
   0.003 mm. Report the initiation time and the mean propagation speed in each case.
2. Keep `u_amp` at 0.002 mm and shorten the ramp to `t_ramp=5.0e-6`. Describe what
   changes in the early part of the von Mises stress animation and explain it in
   terms of the stress wave that the ramp launches.
3. At the highest amplitude, inspect whether the crack remains a single band.
   Comment on what a branching pattern would require of the mesh.

**Your answer:**

_Write here._

### Exercise 5 — Add a field of your own

`RunViewer.series` and `RunViewer.frame` accept any key of `phast.FIELD_REGISTRY`. Choose one
that the notebook has not used, for example `stress_triaxiality` or
`max_principal_strain`.

1. Plot it at three times: before initiation, during propagation and at the end.
2. Animate it and state what it reveals that the four fields in Section 10 do not.
3. Write a short function `band_width(view, x, step)` that returns the damage band
   width at a given station, following the profile in Exercise 1, and use it to
   plot band width against $x$ for the reference run.

**Your answer:**

_Write here._

In [ ]:
print("Fields available through phast.compute_field:")
print(sorted(phast.FIELD_REGISTRY.keys()))

## 13. Where a learned damage model attaches

This interface replaces the damage update within the staggered route.
Geometry, mesh, mechanics, the energy split, the history update, the boundary
conditions and the output all remain with PhAST. The replacement seam lives in
`phast.learned_damage` and is selected by one entry in the solver block:

```yaml
solver:
  damage_update: learned_proposal
  damage_predictor: my_package.damage_adapter:create_predictor
  damage_checkpoint: checkpoints/model.pt
  damage_predictor_options:
    representation: damage_increment
  damage_residual_rtol: 1.0e-3
  damage_fallback: true
```

Two modes exist. `learned_proposal` projects the prediction and uses it as the
initial guess for the classical solve, so the accepted state is still determined
by the classical damage equation. `learned_replacement` accepts the prediction
directly, and only after shape, finiteness, bound, irreversibility, Dirichlet and
projected-residual checks have passed; a rejected prediction falls back to the
classical solver when fallback is enabled.

Assess learned updates using field differences, equation residuals and matched
complete runtimes. Include inference, constraint checks, rejected predictions
and classical fallback in the timing.

Reference material: [`examples/learned_damage`](https://github.com/CEMS-Lab/PhAST/tree/main/examples/learned_damage)
and the [learned damage predictor interface](https://cems-lab.github.io/PhAST/user_guide/learned_damage.html).

## 14. Interpretation and limits

The crack-tip estimate uses a threshold on nodal damage values and is sensitive
to that threshold and the mesh. Report the length scale alongside the mesh
resolution and crack pattern. Record the platform with each timing, including
post-processing and animation export.

Direct learned replacement remains experimental; retain classical fallback when
exploring this interface and inspect rejected updates.

### Further reading

- Borden, Verhoosel, Scott, Hughes and Landis (2012), *A phase-field description of dynamic brittle fracture*, CMAME 217–220, 77–95.
- Miehe, Welschinger and Hofacker (2010), *Thermodynamically consistent phase-field models of fracture*, IJNME 83, 1273–1311.
- PhAST documentation: https://cems-lab.github.io/PhAST/